# Introduction: Meal


In [0]:
%sh
cd ../
pip install . 
# --force-reinstall --no-deps 
pip install s3fs yfinance

In [0]:
import sys
sys.path.append('.')
from DB_MA_finetuning_qkcvlm import *

start_time = time.time()

_v = 2
v_qkcv=1

### Dataset Creation

In [0]:

config_qkcv = Config_qkcv(_v)
config_qkcv.v_qkcv=v_qkcv

config_qkcv.freq_type=1
_col_forecast=config_qkcv.get_col_forecast()
print(_col_forecast)

horizon = 10

### reader
import pyarrow.parquet as pq
import pandas as pd
import s3fs,os
fs = s3fs.S3FileSystem()

_p_base = 's3://'

_p_base_folder = os.path.join(_p_base, 'meal-archive')
file_db=f"Database_db_meal_{_col_forecast}_.csv"

_df_meals = pd.read_csv(os.path.join(_p_base_folder,'meal_info.csv'))
_df_center = pd.read_csv(os.path.join(_p_base_folder,'fulfilment_center_info.csv'))

### features not used in original model
_features_static = ['category', 'cuisine'] + ['city_code', 'region_code', 'center_type', 'op_area']
_features_dynamic = []

_use_staging_train = True # skip data proceeding
_save_to_staging = True # update staging file


In [0]:
import glob

if not _use_staging_train:
    _df_train = pd.read_csv(os.path.join(_p_base_folder, 'train.csv'), sep=',')

    print(f'len {len(_df_train)}')
    _df_train[:10]

    print(f"week.min {_df_train.week.min()}, week.max {_df_train.week.max()}, center_id.max {_df_train.center_id.max()}, num_orders.min {_df_train.num_orders.min()}, num_orders.max() {_df_train.num_orders.max()}")

    unique_id = 'cmid'
    _df_train[unique_id] = _df_train['center_id'] + _df_train['meal_id']*1000

    df_cross_join = _df_train[['week']].drop_duplicates().assign(key=1).merge(_df_train[[unique_id, 'center_id', 'meal_id']].drop_duplicates().assign(key=1), on='key').drop('key', axis=1)

    _df_train['open_flag'] = 1
    _df_train = df_cross_join.merge(_df_train, on=['week', unique_id, 'center_id', 'meal_id'], how='left').sort_values([unique_id, 'week']).reset_index()

    _df_train.count()
    _df_train.isnull().sum()

    _df_train['num_orders'] = _df_train.groupby(unique_id)['num_orders'].ffill()
    _df_train.isnull().sum()

    _df_train['num_orders'].fillna(0.0001, inplace=True)
    _df_train['open_flag'].fillna(0, inplace=True)

    _df_train.isnull().sum()

    _df_train.rename(columns={unique_id: 'unique_id',
                                'num_orders': 'y',
                                }, inplace=True)
    _df_train['ds'] = pd.to_datetime('2020-01-05', format='%Y-%m-%d') + pd.to_timedelta(_df_train['week']*7, unit='d')

    _df_train.columns
    _df_train.head(4)
    _df_train.unique_id.count()
    _df_train.unique_id.nunique()

    _df_complete_numeric = _df_train.copy()

    _df_train = None
    gc.collect()

    for _c in ['y', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured']:
        _df_complete_numeric[_c].fillna(0.0001, inplace=True)

    if _save_to_staging:

        print(f"saving files, y max {_df_complete_numeric['y'].max()}, min {_df_complete_numeric['y'].min()}")
        _df_complete_numeric.ds.max()

        ### save _df_complete_numeric
        num_splits = 1

        # Split the dataframe into smaller dataframes
        df_splits = np.array_split(_df_complete_numeric, num_splits)

        # Save each split into a separate parquet file
        for i, df_split in enumerate(df_splits):
            df_split.to_parquet(os.path.join(_p_base_folder, f'_df_complete_numeric_part_{i}.parquet'))

else:
    # Read all parquet files in the specified directory
    parquet_files =[]
    for i in range(1):
        parquet_files.append(os.path.join(_p_base_folder, f'_df_complete_numeric_part_{i}.parquet'))
    print(f'parquet_files {parquet_files}')
    # Concatenate all the parquet files into a single dataframe
    _df_complete_numeric = pd.concat([pd.read_parquet(file) for file in parquet_files])

_df_complete_numeric.describe()

### _df_static
_df_static_numeric = _df_complete_numeric[['unique_id','meal_id','center_id']].drop_duplicates().merge(_df_meals, on='meal_id', how='left').merge(_df_center, on='center_id', how='left')[_features_static + ['unique_id']]

for _c in _features_static:
    _df_static_numeric[_c] = _df_static_numeric[_c].astype('category').cat.codes


Y_train_df = _df_complete_numeric.loc[(_df_complete_numeric.week >= 20) & (_df_complete_numeric.week < 136), _features_dynamic + ['unique_id', 'ds', 'y']]

Y_test_df = _df_complete_numeric.loc[(_df_complete_numeric.week >= 136) & (_df_complete_numeric.week < 146), _features_dynamic + ['unique_id', 'ds', 'y']]

In [0]:
Y_train_df

### Model Creation

In [0]:
model, hparams, tfm_config = get_model(_features_static, 
                                       config_qkcv,
                                       horizon,
                                       load_weights=True)


In [0]:
predictions, y_future, model_tunned, finetuner, ca, attention_score = prediction_pipeline(tfm_config, horizon, config_qkcv, Y_train_df, Y_test_df, _df_static_numeric, model)


In [0]:
if not config_qkcv.train_only:
    df_merged, pred_vals_tunc = post_predictions(predictions, y_future)

    print(_v)
    print(_col_forecast)
    print(f'{_v}_Meal_{_col_forecast}')

    # Example usage
    wpe_func(df_merged, eval_horizon = [horizon],forecast='forecast')

    # Example usage
    calculate_matrix(df_merged)
    mae = calculate_mae(df_merged)
    print(f"Mean Absolute Error (MAE): {mae}")
    
    print(_v)


In [0]:
print(f"Execution time: {(time.time() - start_time)/60:.2f} mins")

In [0]:
folder_path = f"{_p_base}/results"

filename = f"Meal_{_v}_{_col_forecast}"

if finetuner!=-1:
    pd.DataFrame(finetuner.training_matrix).to_csv(f"{folder_path}/matrix_{filename}.csv", index=False)

df_merged.to_csv(f"{folder_path}/df_merged_{filename}.csv", index=False)

ca.to_csv(f"{folder_path}/ca_{filename}.csv", index=False)
attention_score.to_csv(f"{folder_path}/attention_score_{filename}.csv", index=False)